<a href="https://colab.research.google.com/github/JamesEckhartJr/Quantum-Computing-Projects/blob/main/CALIBRATED_2_QUBIT_PHASE_KICKBACK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import requests
import numpy as np

# ===================== Configuration =====================
DEVICE = 'theq-ae31b1'          # change if needed
URL = f'https://{DEVICE}.quokkacomputing.com/qsim/qasm'
CALIBRATION_SHOTS = 2048
FINAL_SHOTS = 8192
TARGET_PROB = 0.25              # probability we want to hit

print("Calibrating controlled-phase angle on Quokka...\n")

def measure_prob(angle, shots):
    qasm = f"""
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[1];
x q[1];
h q[0];
crz({angle}) q[0], q[1];
h q[0];
measure q[0] -> c[0];
"""
    payload = {"script": qasm.strip(), "count": shots}
    r = requests.post(URL, json=payload, timeout=60)
    shots_data = r.json()["result"]["c"]
    ones = sum(1 for s in shots_data if s[0] == 1)
    return ones / len(shots_data)

# ---------- Calibration sweep ----------
test_angles = np.linspace(0.2, 2.5, 12)
best_angle = None
best_diff = 1.0

print("Angle sweep (calibration):")
for ang in test_angles:
    p = measure_prob(ang, CALIBRATION_SHOTS)
    diff = abs(p - TARGET_PROB)
    print(f"  angle = {ang:.3f} rad  →  P(1) = {p:.3f}")
    if diff < best_diff:
        best_diff = diff
        best_angle = ang

print(f"\nBest calibrated angle: {best_angle:.3f} rad  (closest to target {TARGET_PROB})")

# ---------- Final run with calibrated angle ----------
print("\nRunning final demo with calibrated angle...\n")
final_prob = measure_prob(best_angle, FINAL_SHOTS)

print("=" * 55)
print("CALIBRATED 2-QUBIT PHASE KICKBACK RESULTS")
print("=" * 55)
print(f"Target P(1)         : {TARGET_PROB:.4f}")
print(f"Calibrated angle    : {best_angle:.4f} rad")
print(f"Measured P(1)       : {final_prob:.4f}")
print(f"Absolute error      : {abs(final_prob - TARGET_PROB):.4f}")
print(f"Final shots         : {FINAL_SHOTS}")
print("=" * 55)
print("""
EXPLANATION:
- The Quokka's crz gate uses a phase convention that does not match the
  simple textbook formula sin²(φ/2).
- A short calibration sweep finds an angle that produces the desired
  measurement probability on this specific emulator.
- After calibration the measured probability closely matches the target,
  limited only by shot noise.
- This demonstrates phase kickback while explicitly accounting for the
  backend's gate convention — a practical approach when running on
  real or emulated hardware.
""")

Calibrating controlled-phase angle on Quokka...

Angle sweep (calibration):
  angle = 0.200 rad  →  P(1) = 0.001
  angle = 0.409 rad  →  P(1) = 0.014
  angle = 0.618 rad  →  P(1) = 0.026
  angle = 0.827 rad  →  P(1) = 0.041
  angle = 1.036 rad  →  P(1) = 0.069
  angle = 1.245 rad  →  P(1) = 0.090
  angle = 1.455 rad  →  P(1) = 0.120
  angle = 1.664 rad  →  P(1) = 0.147
  angle = 1.873 rad  →  P(1) = 0.199
  angle = 2.082 rad  →  P(1) = 0.258
  angle = 2.291 rad  →  P(1) = 0.298
  angle = 2.500 rad  →  P(1) = 0.327

Best calibrated angle: 2.082 rad  (closest to target 0.25)

Running final demo with calibrated angle...

CALIBRATED 2-QUBIT PHASE KICKBACK RESULTS
Target P(1)         : 0.2500
Calibrated angle    : 2.0818 rad
Measured P(1)       : 0.2460
Absolute error      : 0.0040
Final shots         : 8192

EXPLANATION:
- The Quokka's crz gate uses a phase convention that does not match the
  simple textbook formula sin²(φ/2).
- A short calibration sweep finds an angle that produces the d